# 🤖 Chatbot Practice (Solution)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nawooooojin/openai-api-tutorial-kr/blob/main/Practice_solution/02-Chatbot-Practice_solution.ipynb)



## 1) 환경 준비

In [ ]:
%pip install -q -U openai ipywidgets

In [1]:
import os

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    try:
        from google.colab import userdata  # type: ignore
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    except ImportError:
        pass  # Colab이 아닌 환경
    except Exception:
        print("Colab 메뉴의 보안 비밀(열쇠 아이콘)에 OPENAI_API_KEY가 저장되어 있는지, 노트북 액세스가 켜져 있는지 확인해주세요.")

if not OPENAI_API_KEY:
    from getpass import getpass
    OPENAI_API_KEY = getpass("OPENAI_API_KEY를 입력하세요: ")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("✅ API 키 설정 완료")

✅ API 키 설정 완료


In [2]:
from openai import OpenAI

client = OpenAI()
print("✅ 클라이언트 생성 완료")

✅ 클라이언트 생성 완료


### OpenAI API 호출 실습

In [3]:
def create_greeting(client, model, instructions, user_prompt):
    """Responses API 호출 솔루션"""
    response = client.responses.create(
        model=model,
        instructions=instructions,
        input=user_prompt,
    )
    return response.output_text

In [4]:
greeting = create_greeting(
    client=client,
    model="gpt-6-luna",
    instructions="You are a helpful assistant. Reply in Korean.",
    user_prompt="오늘 하루를 기분 좋게 시작할 수 있는 인사말을 한 문장으로 만들어 주세요.",
)
print(greeting)

좋은 아침이에요! 오늘은 작은 기쁨이 모여 하루를 환하게 밝혀 주길 바라요.


## 2) 멀티턴 대화 구성 정답

In [5]:
def request_chat_completion(
    client,
    model,
    history,
    reasoning_effort="medium",
    verbosity="medium"
):
    """Responses API 멀티턴 대화 솔루션 구현."""

    kwargs = {}
    if reasoning_effort:
        kwargs["reasoning"] = {"effort": reasoning_effort}
    if verbosity:
        kwargs["text"] = {"verbosity": verbosity}

    # API 호출
    response = client.responses.create(
        model=model,
        input=history,
        **kwargs,
    )
    return response.output_text, response

In [6]:
conversation = [
    {
        "role": "developer",
        "content": "You are a helpful assistant. Reply in Korean and keep answers concise.",
    },
    {
        "role": "user",
        "content": "방금 읽은 책을 한 줄로 요약해 달라고 부탁하는 방법을 알려 줘.",
    },
]

first_text, first_response = request_chat_completion(
    client=client,
    model="gpt-6-luna",
    history=conversation,
    reasoning_effort="medium",
    verbosity="low"
)
print("첫 번째 응답:", first_text)

conversation.append({"role": "assistant", "content": first_text})
conversation.append({"role": "user", "content": "이 내용을 더 자세하게 바꿔 줄래?"})

second_text, second_response = request_chat_completion(
    client=client,
    model="gpt-6-luna",
    history=conversation,
    verbosity="high"
)
print("\n두 번째 응답:", second_text)

첫 번째 응답: “방금 읽은 책의 핵심 내용을 한 문장으로 요약해 줘”라고 부탁하면 돼요. 책 제목을 덧붙이면 더 정확해요.



두 번째 응답: 이렇게 부탁하면 더 구체적이에요.

“방금 읽은 **[책 제목]**의 핵심 주제와 저자가 전하려는 메시지를 중심으로, 책의 중요한 내용을 빠뜨리지 않으면서 한 문장으로 요약해 줘. 책을 읽지 않은 사람도 내용을 이해할 수 있도록 쉽고 명확하게 써 줘.”


## 3) 챗봇 만들기 실습 정답

In [7]:
class ChatSession:
    """Responses API 기반 멀티턴 챗봇 세션 관리 유틸리티 (정답)."""

    def __init__(
        self,
        client: OpenAI,
        model: str = "gpt-6-luna",
        developer_message: str = "You are a helpful assistant. You must answer in Korean and keep answers clear.",
        reasoning_effort: str = "medium",
        verbosity: str = "medium",
    ):
        self.client = client
        self.model = model
        self.developer_message = developer_message
        self.reasoning_effort = reasoning_effort
        self.verbosity = verbosity
        self.history = []
        self.last_response_id = None
        self.reset()

    def reset(self) -> None:
        """대화 히스토리를 초기 developer 메시지로 리셋합니다."""
        self.history = [
            {
                "role": "developer",
                "content": self.developer_message,
            }
        ]
        self.last_response_id = None

    def send(self, user_message: str):
        """사용자 메시지를 추가하고 모델 응답 텍스트와 response 객체를 반환합니다."""
        cleaned = user_message.strip()
        if not cleaned:
            raise ValueError("user_message는 비어 있을 수 없습니다.")

        self.history.append({"role": "user", "content": cleaned})

        message_text, response = request_chat_completion(
            client=self.client,
            model=self.model,
            history=self.history,
            reasoning_effort=self.reasoning_effort,
            verbosity=self.verbosity,
        )

        self.history.append({"role": "assistant", "content": message_text})
        self.last_response_id = response.id
        return message_text, response

    def pretty_history(self):
        """현재 히스토리를 읽기 쉬운 문자열로 변환합니다."""
        lines = []
        for msg in self.history:
            prefix = "사용자" if msg["role"] == "user" else "assistant" if msg["role"] == "assistant" else msg["role"]
            lines.append(f"[{prefix}] {msg['content']}")
        return "\n".join(lines)

In [8]:
chatbot = ChatSession(
    client,
    model="gpt-6-luna",
    developer_message="You are a helpful teaching assistant. Reply in Korean with actionable suggestions.",
    reasoning_effort="low",
)

In [9]:
reply, raw_response = chatbot.send("Responses API 실습을 복습하는 가장 좋은 방법을 알려 줘.")
print("모델 응답:\n", reply)

print("\n현재 히스토리:\n", chatbot.pretty_history())

모델 응답:
 Responses API를 복습할 때는 문서를 다시 읽기만 하기보다 **직접 호출하고, 결과를 설명하고, 작은 기능을 만들어 보는 방식**이 가장 효과적입니다.

1. **핵심 요청을 처음부터 다시 작성해 보세요.**  
   간단한 입력을 보내고 응답을 확인하면서 `input`, `instructions`, 모델 지정 등 실습에서 사용한 요소가 각각 어떤 역할을 하는지 설명해 보세요.

2. **응답을 직접 살펴보세요.**  
   결과 텍스트만 보지 말고 응답의 구조와 사용한 출력 항목도 확인하세요. “모델이 무엇을 반환했고, 내 코드가 그중 무엇을 사용했나?”를 정리하면 디버깅에도 도움이 됩니다.

3. **기능을 하나씩 추가해 보세요.**  
   기본 요청이 익숙해지면 실습에서 다룬 기능을 하나씩 붙여 보세요. 예를 들어 대화 맥락 유지, 도구 호출, 구조화된 출력 등이 있다면 기능별로 작은 예제를 만드세요. 한 번에 여러 기능을 추가하면 오류 원인을 찾기 어렵습니다.

4. **일부러 실패시켜 보세요.**  
   잘못된 입력, 누락된 설정, 예상과 다른 출력 등을 시험하고 오류 메시지와 응답을 살펴보세요. 실습 내용을 실제로 이해했는지 확인하기 좋은 방법입니다.

5. **작은 프로젝트로 마무리하세요.**  
   예를 들어 메모 요약기나 질문 답변기를 만들고, 입력 → Responses API 요청 → 결과 처리 과정을 직접 구현해 보세요. 코드를 보지 않고 핵심 흐름을 다시 작성할 수 있으면 복습이 잘 된 것입니다.

마지막으로 실습별로 **“요청에 넣은 값 / 응답에서 확인한 값 / 새로 배운 점”**을 세 줄로 기록해 두세요. 다음 복습 때는 먼저 노트 없이 재현해 보고, 막히는 부분만 문서와 비교하면 시간을 아낄 수 있습니다. API의 매개변수나 동작은 바뀔 수 있으니, 실제 구현 전에는 최신 공식 문서도 확인하세요.

현재 히스토리:
 [developer] You are a helpful teaching assistant.

## 4) 위젯 기반 챗봇 UI (정답)

In [10]:
import ipywidgets as widgets
from IPython.display import Markdown, display

chat_ui_session = ChatSession(
    client,
    model="gpt-6-luna",
    developer_message="You are a supportive study buddy. Reply in Korean with friendly tips.",
    reasoning_effort="medium",
)

log_output = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", padding="8px"))
input_box = widgets.Text(
    placeholder="질문을 입력하고 Enter 또는 '전송'을 클릭하세요.",
    description="질문",
    layout=widgets.Layout(width="100%"),
)
send_button = widgets.Button(description="전송", button_style="primary")
reset_button = widgets.Button(description="초기화")
status_label = widgets.HTML(value="")


def append_message(role: str, text: str) -> None:
    with log_output:
        display(Markdown(f"**{role}**: {text}"))


def handle_send(_=None) -> None:
    text = input_box.value.strip()
    if not text:
        status_label.value = "<span style='color:#d9534f;'>메시지를 입력하세요.</span>"
        return

    status_label.value = ""
    append_message("사용자", text)
    input_box.value = ""
    send_button.disabled = True
    try:
        answer, _ = chat_ui_session.send(text)
        append_message("어시스턴트", answer)
    except Exception as exc:  # pragma: no cover
        append_message("오류", f"{type(exc).__name__}: {exc}")
    finally:
        send_button.disabled = False


def handle_reset(_=None) -> None:
    chat_ui_session.reset()
    log_output.clear_output()
    status_label.value = "<span style='color:#5cb85c;'>세션이 초기화되었습니다.</span>"


send_button.on_click(handle_send)
reset_button.on_click(handle_reset)

input_box.continuous_update = False
input_box.observe(lambda change: handle_send() if change['new'].strip() else None, names='value')

controls = widgets.HBox([send_button, reset_button, status_label])
ui = widgets.VBox([input_box, controls, log_output])

display(ui)

